# Benchmark Data Explorer

This notebook explores the merged benchmark data for Qwen 3.5 models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

df = pd.read_parquet("merged_benchmark_data.parquet")
print(f"Shape: {df.shape}")
df.head()

## Dataset Overview

In [ ]:
print("=== Column Names ===")
print(df.columns.tolist())
print("\n=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())

In [ ]:
import re
model_pattern = re.compile(r'^(.+?)_(solved|solved_norm|energy_joules|latency_ms|throughput|prompt_tokens|completion_tokens|total_tokens)$')
models = []
for c in df.columns:
    match = model_pattern.match(c)
    if match:
        models.append(match.group(1))
models = sorted(set(models))
print(f"Models: {models}")
print(f"\nBenchmarks: {sorted(df['benchmark'].unique())}")
print(f"\nTotal unique documents: {df['doc_id'].nunique()}")

## Descriptive Statistics by Benchmark

In [ ]:
for model in models:
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print('='*60)
    
    solved_col = f"{model}_solved"
    energy_col = f"{model}_energy_joules"
    latency_col = f"{model}_latency_ms"
    throughput_col = f"{model}_throughput"
    
    stats_data = []
    for benchmark in sorted(df['benchmark'].unique()):
        bench_df = df[df['benchmark'] == benchmark]
        
        row = {
            'benchmark': benchmark,
            'n_samples': len(bench_df),
            'solved_mean': bench_df[solved_col].mean() if solved_col in bench_df.columns else np.nan,
            'solved_std': bench_df[solved_col].std() if solved_col in bench_df.columns else np.nan,
            'energy_mean': bench_df[energy_col].mean() if energy_col in bench_df.columns else np.nan,
            'energy_std': bench_df[energy_col].std() if energy_col in bench_df.columns else np.nan,
            'latency_mean': bench_df[latency_col].mean() if latency_col in bench_df.columns else np.nan,
            'latency_std': bench_df[latency_col].std() if latency_col in bench_df.columns else np.nan,
            'throughput_mean': bench_df[throughput_col].mean() if throughput_col in bench_df.columns else np.nan,
        }
        stats_data.append(row)
    
    stats_df = pd.DataFrame(stats_data)
    print(stats_df.to_string(index=False))

## Solved Rate Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

solved_data = []
for model in models:
    solved_col = f"{model}_solved"
    for benchmark in sorted(df['benchmark'].unique()):
        bench_df = df[df['benchmark'] == benchmark]
        if solved_col in bench_df.columns:
            solved_data.append({
                'model': model,
                'benchmark': benchmark,
                'solved_rate': bench_df[solved_col].mean()
            })

solved_df = pd.DataFrame(solved_data)
solved_pivot = solved_df.pivot(index='benchmark', columns='model', values='solved_rate')
solved_pivot.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Benchmark')
ax.set_ylabel('Solved Rate')
ax.set_title('Solved Rate by Benchmark and Model')
ax.set_ylim(0, 1)
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Energy Consumption by Benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

energy_data = []
for model in models:
    energy_col = f"{model}_energy_joules"
    for benchmark in sorted(df['benchmark'].unique()):
        bench_df = df[df['benchmark'] == benchmark]
        if energy_col in bench_df.columns:
            energy_data.append({
                'model': model,
                'benchmark': benchmark,
                'energy_joules': bench_df[energy_col].mean()
            })

energy_df = pd.DataFrame(energy_data)
energy_pivot = energy_df.pivot(index='benchmark', columns='model', values='energy_joules')
energy_pivot.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Benchmark')
ax.set_ylabel('Mean Energy (Joules)')
ax.set_title('Mean Energy Consumption by Benchmark and Model')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Latency by Benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

latency_data = []
for model in models:
    latency_col = f"{model}_latency_ms"
    for benchmark in sorted(df['benchmark'].unique()):
        bench_df = df[df['benchmark'] == benchmark]
        if latency_col in bench_df.columns:
            latency_data.append({
                'model': model,
                'benchmark': benchmark,
                'latency_ms': bench_df[latency_col].mean()
            })

latency_df = pd.DataFrame(latency_data)
latency_pivot = latency_df.pivot(index='benchmark', columns='model', values='latency_ms')
latency_pivot.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Benchmark')
ax.set_ylabel('Mean Latency (ms)')
ax.set_title('Mean Latency by Benchmark and Model')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Throughput by Benchmark

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

throughput_data = []
for model in models:
    throughput_col = f"{model}_throughput"
    for benchmark in sorted(df['benchmark'].unique()):
        bench_df = df[df['benchmark'] == benchmark]
        if throughput_col in bench_df.columns:
            throughput_data.append({
                'model': model,
                'benchmark': benchmark,
                'throughput': bench_df[throughput_col].mean()
            })

throughput_df = pd.DataFrame(throughput_data)
throughput_pivot = throughput_df.pivot(index='benchmark', columns='model', values='throughput')
throughput_pivot.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Benchmark')
ax.set_ylabel('Mean Throughput (tokens/sec)')
ax.set_title('Mean Throughput by Benchmark and Model')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Token Usage Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, token_type in enumerate(['prompt_tokens', 'completion_tokens', 'total_tokens']):
    ax = axes[idx]
    token_data = []
    for model in models:
        col = f"{model}_{token_type}"
        if col in df.columns:
            model_data = df[[col, 'benchmark']].dropna()
            model_data = model_data.copy()
            model_data['model'] = model
            token_data.append(model_data)
    
    if token_data:
        token_df = pd.concat(token_data)
        sns.boxplot(data=token_df, x='benchmark', y=col, hue='model', ax=ax)
        ax.set_title(f'{token_type.replace("_", " ").title()}')
        ax.set_xlabel('Benchmark')
        ax.set_ylabel(token_type.replace('_', ' ').title())
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Correlation: Solved Rate vs Energy/Latency

In [ ]:
fig, axes = plt.subplots(len(models), 2, figsize=(12, 4 * len(models)))

for i, model in enumerate(models):
    solved_col = f"{model}_solved"
    energy_col = f"{model}_energy_joules"
    latency_col = f"{model}_latency_ms"
    
    bench_stats = df.groupby('benchmark').agg({
        solved_col: 'mean',
        energy_col: 'mean',
        latency_col: 'mean'
    }).reset_index()
    
    if i < len(models):
        ax = axes[i, 0] if len(models) > 1 else axes[0]
        ax.scatter(bench_stats[solved_col], bench_stats[energy_col])
        ax.set_xlabel('Solved Rate')
        ax.set_ylabel('Energy (Joules)')
        ax.set_title(f'{model}: Solved Rate vs Energy')
        for j, row in bench_stats.iterrows():
            ax.annotate(row['benchmark'], (row[solved_col], row[energy_col]), fontsize=8)
        
        ax = axes[i, 1] if len(models) > 1 else axes[1]
        ax.scatter(bench_stats[solved_col], bench_stats[latency_col])
        ax.set_xlabel('Solved Rate')
        ax.set_ylabel('Latency (ms)')
        ax.set_title(f'{model}: Solved Rate vs Latency')
        for j, row in bench_stats.iterrows():
            ax.annotate(row['benchmark'], (row[solved_col], row[latency_col]), fontsize=8)

plt.tight_layout()
plt.show()

## Model Comparison Summary

In [ ]:
summary_data = []
for model in models:
    solved_col = f"{model}_solved"
    energy_col = f"{model}_energy_joules"
    latency_col = f"{model}_latency_ms"
    throughput_col = f"{model}_throughput"
    
    summary_data.append({
        'model': model,
        'overall_solved_rate': df[solved_col].mean(),
        'mean_energy_joules': df[energy_col].mean(),
        'mean_latency_ms': df[latency_col].mean(),
        'mean_throughput': df[throughput_col].mean(),
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))